## Initialization

### Imports/Constants

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    # TypeVar,
    # Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log
import shutil

import matplotlib.pyplot as plt
import matplotlib as mpl
import mpl_toolkits.mplot3d.art3d as art3d
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from pint import Quantity

from data_processing.paths import (
    get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    NonReactorDataframeColumn,
    SliceFitDataframeColumn,
    EnergyColumn,
    get_df_col
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.loading.dataframe_loading import load_parquet_psd
from data_processing.loading.timetag_processing import (
    calculate_timetag_hours,
    calculate_event_time
)
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
# from data_processing.processing.slice_fitting import (
#     get_psd_energy_histogram, scan_histogram_slices, find_failed_slices)
# from data_processing.processing.calibration import Detector, recalibrate
# from data_processing.processing.neutron_classification import classify
from data_processing import processing as proc
from data_processing import types as proc_types
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing import helpers
from data_processing.processing.neutron_window_strategy.strategy_factory import \
    NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import \
    AbstractNeutronStrategy
from data_processing.types import (
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    WindowType,
    SliceFitStyle,
    BimodalBounds,
    BimodalParams
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

### Functions

In [ ]:
def bin_non_neutron_data(df, time_bins, data_col, selected_cols):
    start_time = time_bins[0]
    df = get_time_cut(df, 'Time', time_bins)

    binned_df = df.groupby("Time Bin", as_index=False)[data_col] \
        .agg(['mean', 'std']) \
        .copy()
    binned_df.columns = selected_cols
    binned_df['Bin midpoint'] = binned_df.index.to_series() \
        .apply(lambda x: x.mid)
    binned_df = bin_midpoint_time_to_seconds(binned_df, start_time)

    return binned_df

In [ ]:
def bin_midpoint_time_to_seconds(df, start_time):
    bin_mid_col = df[BinningDataframeColumn.BIN_MIDPOINT.value]
    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
    zeroed_midpoint = pd.to_datetime(bin_mid_col) - start_time
    df[bin_time_col_name] = zeroed_midpoint.dt.total_seconds()
    return df

In [ ]:
def get_time_cut(df, time_tag_col, time_bins):
    timetag_cut = pd.cut(df[time_tag_col], bins=time_bins)
    df[BinningDataframeColumn.TIME_BIN.value] = timetag_cut
    return df

In [ ]:
# fns ask questions, then generate strategy using factory

CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]


def get_nasa_loading_settings(
    calib_key: CalibrationKey
) -> str:
    left_border_type = helpers.get_input_with_default(
        """\
Which left border calculation do you want to use?
1: original left border (0.1966 MeVee)
2: newer left border (~0.1866 MeVee)
3: CAEN lower limit (0.050 MeVee) (default)
Press Enter for default
""",
        3,
        int
    )
    border_key: NasaBorderKey = (
        ExperimentDataKey.NASA_BORDERS if left_border_type == 1 
        else ExperimentDataKey.NASA_BORDERS_RECALC
    )
    file_name_prefix = f"{calib_key.value}_{border_key.value}"
    return file_name_prefix


def get_n_distro_loading_settings(
    calib_key: CalibrationKey
) -> str:
    file_name_prefix = f"{calib_key.value}_{ExperimentDataKey.N_WINDOW_BORDERS.value}"
    return file_name_prefix


def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> NasaGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = helpers.get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = helpers.get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = helpers.get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee)
2: newer (~0.1866 MeVee)
3: detector lower limit (0.050 MeVee) (default)
or press Enter for default
""",
            3,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = (
                ExperimentDataKey.NASA_BORDERS 
                if existing_left_border_version_input == 1 
                else ExperimentDataKey.NASA_BORDERS_RECALC
            )
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = get_neutron_window_paths(
                file_name_prefix=file_name_prefix)
            left_border, _ = load_side_borders(
                side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        elif existing_left_border_version_input == 3:
            lower_energy_bound = 0.05
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = helpers.get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.050)
""",
            0.050,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def get_n_distro_generation_settings(
) -> NeutronDistributionGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (3)
""",
        3,
        float
    )
    settings = NeutronDistributionGenerationSettings(
        sigma=sigma
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: NeutronStrategyFactory,
    window_type: WindowType,
    loading: bool,
    settings: NeutronWindowSettings
) -> Callable[[], AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data

In [ ]:
def get_psd_adc_histogram(
    df: pd.DataFrame,
    adc_width: float = 420,
    adc_bins: np.ndarray | None = None,
    psd_bin_count: int = 100,
    psd_min: float = 0.0,
    psd_max: float = 0.5
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x = get_df_col(df, DetectorDataframeColumn.ENERGY)
    y = get_df_col(df, DetectorDataframeColumn.PSD)

    within_psd = y.between(psd_min, psd_max)
    x = x[within_psd == True].copy()
    y = y[within_psd == True].copy()

    if adc_bins is not None:
        x_bins = adc_bins
    else:
        x_bins: np.ndarray = np.linspace(
            0, x.max(), int(x.max() / adc_width) + 1
        )
    print(f"Energy width = {x_bins[1]-x_bins[0]} ADC")
    y_bins: np.ndarray = np.linspace(psd_min, psd_max, psd_bin_count + 1)

    Z, xe, ye = np.histogram2d(x, y, bins=[x_bins, y_bins])
    return Z, xe, ye

## Experiment ID Input

In [ ]:
experiment_ids = ["TB-26"]

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
window_offset = 0.2
sigma = 5
lower_energy_bound = 0.05
recalc_lower_bound = False
settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

## Data Loading and Initial Processing

### Data Processing

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load_parquet_psd(exp_id)

In [ ]:
# Generate histogram

adc_width = 20
# adc_width = 100
# overall_settings['scan_idx'] = f"({start_scan_idx}, {end_scan_idx})"
# overall_settings['energy_width'] = energy_width

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = get_psd_adc_histogram(
        psd_report,
        adc_width=adc_width,
        # psd_bin_count=50
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    # exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

## Plotting

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"

In [ ]:
def lerp(x: float, p1: tuple[float, float], p2: tuple[float, float]) -> float:
    x1, y1 = p1
    x2, y2 = p2
    m = (y2 - y1) / (x2 - x1)
    return m * (x - x1) + y1


def clamp(x: float, xmin: float, xmax: float) -> float:
    if x < xmin:
        return xmin
    elif x > xmax:
        return xmax
    else:
        return x


def to_unit_interval(x: float, xmin: float, xmax: float) -> float:
    return clamp(lerp(x, (xmin, 0), (xmax, 1)), 0, 1)


from typing import TypeGuard, TypeVar, Any


ColorHexCode = str  # of format #xxxxxx, can be made into hex value
ColorAlphaHexCode = str  # of format #xxxxxxxx, can be made into hex value
ColorTuple = tuple[float, float, float]
ColorAlphaTuple = tuple[float, float, float, float]
ColorType = ColorHexCode | ColorAlphaHexCode | ColorTuple | ColorAlphaTuple
CT = TypeVar("CT", bound=ColorType)


def is_color_hex_code(x: Any) -> TypeGuard[ColorHexCode]:
    if not isinstance(x, str):
        return False
    pattern_match = re.match(r"#[0-9a-fA-F]{6}$", x)
    return pattern_match is not None


def is_color_alpha_hex_code(x: Any) -> TypeGuard[ColorAlphaHexCode]:
    if not isinstance(x, str):
        return False
    pattern_match = re.match(r"#[0-9a-fA-F]{8}$", x)
    return pattern_match is not None


def is_color_tuple(x: Any) -> TypeGuard[ColorTuple]:
    if not isinstance(x, tuple):
        return False
    if not len(x) == 3:
        return False
    elements_in_limits = [0 <= elem <= 1 for elem in x]
    return all(elements_in_limits)


def is_color_alpha_tuple(x: Any) -> TypeGuard[ColorAlphaTuple]:
    if not isinstance(x, tuple):
        return False
    if not len(x) == 4:
        return False
    elements_in_limits = [0 <= elem <= 1 for elem in x]
    return all(elements_in_limits)


def to_color_float(x: str) -> float:
    try:
        color = int(x, base=16) / 255
    except ValueError:
        raise ValueError(f"{x} is not valid hexadecimal code")
    if color < 0 or color > 1:
        raise ValueError(f"{x} is out of color float bounds (0-1)")
    return color


def to_hex_chars(x: float) -> str:
    if x < 0 or x > 1:
        raise ValueError(f"{x} is out of color float bounds (0-1)")
    x_int = min(255, int(x * 256))
    return f"{x_int:02X}"


def to_color_alpha_tuple(x: ColorType) -> ColorAlphaTuple:
    if is_color_alpha_tuple(x):
        return x
    elif is_color_alpha_hex_code(x):
        r = to_color_float(x[1:3])
        g = to_color_float(x[3:5])
        b = to_color_float(x[5:7])
        a = to_color_float(x[7:9])
        return (r, g, b, a)
    elif is_color_tuple(x):
        r, g, b = x
        return (r, g, b, 1)
    elif is_color_hex_code(x):
        r = to_color_float(x[1:3])
        g = to_color_float(x[3:5])
        b = to_color_float(x[5:7])
        return (r, g, b, 1)
    else:
        raise ValueError(f"{x} could not be converted to ColorAlphaTuple")


def to_color_tuple(x: ColorType) -> ColorTuple:
    if is_color_alpha_tuple(x):
        r, g, b, _ = x
        return (r, g, b)
    elif is_color_alpha_hex_code(x):
        r = to_color_float(x[1:3])
        g = to_color_float(x[3:5])
        b = to_color_float(x[5:7])
        return (r, g, b)
    elif is_color_tuple(x):
        return x
    elif is_color_hex_code(x):
        r = to_color_float(x[1:3])
        g = to_color_float(x[3:5])
        b = to_color_float(x[5:7])
        return (r, g, b)
    else:
        raise ValueError(f"{x} could not be converted to ColorTuple")


def to_color_hex_code(x: ColorType) -> ColorHexCode:
    if is_color_alpha_tuple(x):
        r, g, b, _ = x
        r_hex = to_hex_chars(r)
        g_hex = to_hex_chars(g)
        b_hex = to_hex_chars(b)
        return ("#" + r_hex + g_hex + b_hex).lower()
    elif is_color_alpha_hex_code(x):
        return x[:-2]
    elif is_color_tuple(x):
        r, g, b = x
        r_hex = to_hex_chars(r)
        g_hex = to_hex_chars(g)
        b_hex = to_hex_chars(b)
        return ("#" + r_hex + g_hex + b_hex).lower()
    elif is_color_hex_code(x):
        return x
    else:
        raise ValueError(f"{x} could not be converted to ColorHexCode")


def to_color_alpha_hex_code(x: ColorType) -> ColorAlphaHexCode:
    if is_color_alpha_tuple(x):
        r, g, b, a = x
        r_hex = to_hex_chars(r)
        g_hex = to_hex_chars(g)
        b_hex = to_hex_chars(b)
        a_hex = to_hex_chars(a)
        return ("#" + r_hex + g_hex + b_hex + a_hex).lower()
    elif is_color_alpha_hex_code(x):
        return x
    elif is_color_tuple(x):
        r, g, b = x
        r_hex = to_hex_chars(r)
        g_hex = to_hex_chars(g)
        b_hex = to_hex_chars(b)
        return ("#" + r_hex + g_hex + b_hex + "ff").lower()
    elif is_color_hex_code(x):
        return x+"ff"
    else:
        raise ValueError(f"{x} could not be converted to ColorAlphaHexCode")


def calculate_gradient_color(
    gradient_unit_interval: float,
    color_from: ColorAlphaTuple,
    color_to: ColorAlphaTuple
) -> ColorAlphaTuple:
    if gradient_unit_interval < 0 or gradient_unit_interval > 1:
        raise ValueError(f"{x} is out of bounds (0-1)")
    grad_color = tuple([
        clamp(lerp(gradient_unit_interval, (0, val_from), (1, val_to)), val_from, val_to)
        for val_from, val_to in zip(color_from, color_to)
    ])
    return grad_color

In [ ]:
cmap = plt.colormaps["nipy_spectral"]
figsize = (12, 12)
fontsize = 16
histo_res = 128
contour_res = 100
angle_elev = 30
angle_rot = -30


def get_gamma_neutron_color(
    is_gamma: bool,
    is_neutron: bool,
    is_uncertain: bool,
    psd: float,
    gamma_color: ColorAlphaTuple,
    neutron_color: ColorAlphaTuple
) -> ColorAlphaTuple | None:
    if is_uncertain:
        psd_unit_interval = to_unit_interval(psd, 0.2, 0.3)
        return calculate_gradient_color(psd_unit_interval, gamma_color, neutron_color)
    elif is_gamma:
        return gamma_color
    elif is_neutron:
        return neutron_color
    else:
        return None


for exp_name, data_dict in experiment_neutron_data.items():
    Z = data_dict[ExperimentDataKey.PSD_HISTOGRAM]
    Z = Z.T
    Z_rows, Z_cols = Z.shape
    xe = data_dict[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = data_dict[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    fig = plt.figure(figsize=figsize)
    ax = plt.axes(projection='3d')
    x, y = np.meshgrid(xe[:-1], ye[:-1])
    
    # TODO make colors grid
    # if PSD 0-0.25, Z > ? -> gamma -> grey
    # if PSD 0.25-0.5, Z > ? -> neutron -> blue
    # if PSD 0.2-0.3, E < 500, Z < ? -> either? -> gradient
    # TODO figure out gradient - based on PSD value (high=blue, low=grey) and Z (less=more transparent)
    # can do low Z transparency using lerp (alpha 0 for Z < 10, alpha 1 for Z > ?, interpolate between)
    # gradient_zone = (y >= 0.2) & (y < 0.3) & (x < 500) & (Z < 2000)
    gradient_zone = np.full_like(y, False, dtype=bool)
    neutron_psd = (y >= 0) & (y < 0.25) & (~gradient_zone)
    gamma_psd = (y >= 0.25) & (y < 0.5) & (~gradient_zone)
    
    alphas = [
        [
            # clamp(lerp(cell, (10, 0), (1000, 1)), 0, 1)
            to_unit_interval(cell, 10, 1000)
            for cell in row
        ] for row in Z
    ]
    bg_blue_ct = to_color_alpha_tuple(bg_blue)
    bg_grey_ct = to_color_alpha_tuple(bg_grey)
    colors = [
        [
            get_gamma_neutron_color(
                gamma_psd[i, j],
                neutron_psd[i, j],
                gradient_zone[i, j],
                y[i, j],
                bg_grey_ct,
                bg_blue_ct
            ) for j in range(Z_cols)
        ] for i in range(Z_rows)
    ]
    color_alphas = [
        [
            (0, 0, 0, 0) if color is None else (color[0], color[1], color[2], alpha)
            for color, alpha in zip(colors_row, alphas_row)
        ] for colors_row, alphas_row in zip(colors, alphas)
    ]

    ax.view_init(angle_elev, angle_rot)
    ax.plot_surface(x, y, Z, facecolors=color_alphas)
    # ax.contour3D(x, y, Z, contour_res, cmap=cmap, linewidths=1)
    ax.plot_wireframe(x, y, Z, linewidths=1, alpha=0.25)
    # ax.plot(x, y, Z, "o", lw=0)

    bbox_params = {"boxstyle": "round", "fc": "white", "lw": 0, "fill": True, "alpha": 0.9}
    text_params = {"fontsize": fontsize-2, "bbox": bbox_params, "zorder": 5, "va": "center"}
    # ax.text(0, 0.10, 2000, "Gamma", **text_params)
    # ax.text(0, 0.32, 3000, "Neutron", **text_params)
    
    # ax.set_title(f"{exp_name} PSD/Energy 3D Histogram", fontsize=fontsize+4)
    ax.set_xlabel("Energy (ADC channel x1000)", fontsize=fontsize)
    ax.set_ylabel("PSD", fontsize=fontsize)
    ax.set_zlabel("Counts (x1000)", fontsize=fontsize)
    ax.xaxis.set_major_formatter(lambda x, pos: f"{x / 1000:.1f}")
    ax.zaxis.set_major_formatter(lambda z, pos: f"{z / 1000:.1f}")
    ax.tick_params(labelsize=fontsize)
    ax.tick_params(pad=0)
    ax.tick_params(axis="x", pad=2)
    ax.tick_params(axis="z", pad=-2)
    for axis3d in [ax.xaxis, ax.yaxis, ax.zaxis]:
        axis3d.labelpad = 12
    ax.zaxis.labelpad = 14
    xaxis_ticklabels = ax.xaxis.get_ticklabels()
    for ticklabel in xaxis_ticklabels:
        ticklabel.set_ha("right")
        ticklabel.set_va("bottom")
    yaxis_ticklabels = ax.yaxis.get_ticklabels()
    for ticklabel in yaxis_ticklabels:
        ticklabel.set_ha("left")
        ticklabel.set_va("top")
    zaxis_ticklabels = ax.zaxis.get_ticklabels()
    for ticklabel in zaxis_ticklabels:
        ticklabel.set_ha("left")
        ticklabel.set_va("center")
    ax.set_box_aspect(None, zoom=0.85)

In [ ]:
def get_gamma_neutron_color(
    is_gamma: bool,
    is_neutron: bool,
    is_uncertain: bool,
    psd: float,
    gamma_color: ColorAlphaTuple,
    neutron_color: ColorAlphaTuple
) -> ColorAlphaTuple | None:
    if is_uncertain:
        psd_unit_interval = to_unit_interval(psd, 0.2, 0.3)
        return calculate_gradient_color(psd_unit_interval, gamma_color, neutron_color)
    elif is_gamma:
        return gamma_color
    elif is_neutron:
        return neutron_color
    else:
        return None


def text3d(ax, xyz, s, zdir="z", size=None, angle=0, usetex=False, **kwargs):
    """
    Plots the string *s* on the Axes *ax*, with position *xyz*, size *size*,
    and rotation angle *angle*. *zdir* gives the axis which is to be treated as
    the third dimension. *usetex* is a boolean indicating whether the string
    should be run through a LaTeX subprocess or not.  Any additional keyword
    arguments are forwarded to `.transform_path`.

    Originally created by matplotlib development team.
    See https://matplotlib.org/stable/gallery/mplot3d/pathpatch3d.html

    Note: zdir affects the interpretation of xyz.
    """
    x, y, z = xyz
    if zdir == "y":
        xy1, z1 = (x, z), y
    elif zdir == "x":
        xy1, z1 = (y, z), x
    else:
        xy1, z1 = (x, y), z

    text_path = mpl.text.TextPath((0, 0), s, size=size, usetex=usetex)
    trans = mpl.transforms.Affine2D().rotate(angle).translate(xy1[0], xy1[1])

    p1 = mpl.patches.PathPatch(trans.transform_path(text_path), **kwargs)
    ax.add_patch(p1)
    art3d.pathpatch_2d_to_3d(p1, z=z1, zdir=zdir)


def get_bbox_center(bbox: mpl.transforms.BboxBase) -> tuple[float, float]:
    return (bbox.xmin + bbox.width / 2, bbox.ymin + bbox.height / 2)


def get_translation_to(
    pos_from: tuple[float, float],
    pos_to: tuple[float, float]
) -> tuple[float, float]:
    x_from, y_from = pos_from
    x_to, y_to = pos_to
    return (x_to - x_from, y_to - y_from)


def make_text_path(
    text: str | list[str],
    # position: tuple[float, float],
    # rotation: float,
    # scaling: tuple[float, float],
    linespacing: float,
    font_properties=None,
    usetex=False
) -> mpl.text.TextPath:
    # scaling_x, scaling_y = scaling
    # pos_x, pos_y = position
    paths = []
    
    if isinstance(text, str):
        text = [text]
    
    for i, line in enumerate(text):
        text_path = mpl.text.TextPath(
            (0, 0), line,
            prop=font_properties,
            size=1,
            usetex=usetex
        )
        bbox = text_path.get_extents()
        line_width = bbox.width
        from_x = bbox.xmin
        from_y = bbox.xmax
        # center first line on x=0, anchor at y=0, next lines below
        to_x = -line_width / 2
        to_y = -i * linespacing
        x = to_x - from_x
        y = to_y - from_y
        
        line_transform = mpl.transforms.Affine2D().translate(x, y)
        transformed_path = line_transform.transform_path(text_path)
        paths.append(transformed_path)
    
    text_path = mpl.path.Path.make_compound_path(*paths)
    return text_path


def make_bounding_box_for_text_path(
    text_path: mpl.text.TextPath,
    padding: float,
    rounding_size: float,
    zorder: int,
    mutation_aspect: float = 1,
    **bbox_params
) -> mpl.patches.FancyBboxPatch:
    text_bbox = text_path.get_extents()
    # get anchor corner, width, height
    # make FancyBboxPatch
    anchor = text_bbox.min
    boxstyle = f"round, pad={padding}, rounding_size={rounding_size}"
    return mpl.patches.FancyBboxPatch(
        anchor,
        text_bbox.width,
        text_bbox.height,
        boxstyle=boxstyle,
        mutation_aspect=mutation_aspect,
        **bbox_params
    )


def convert_text_path_to_patch(
    text_path: mpl.text.TextPath,
    zorder: int
) -> mpl.patches.PathPatch:
    return mpl.patches.PathPatch(text_path, ec="none", fc="k", zorder=zorder)


def patch_to_3d_plot_wall(
    patch: mpl.patches.Patch,
    ax: mpl.axes.Axes,
    zdir: str,
    z: float = 0,
):
    ax.add_patch(patch)
    art3d.pathpatch_2d_to_3d(patch, z=z, zdir=zdir)


def get_center_match_transform(
    box_from: mpl.transforms.Bbox,
    box_to: mpl.transforms.Bbox
) -> tuple[float, float]:
    from_c_x = (box_from.x0 + box_from.x1) / 2
    from_c_y = (box_from.y0 + box_from.y1) / 2
    to_c_x = (box_to.x0 + box_to.x1) / 2
    to_c_y = (box_to.y0 + box_to.y1) / 2
    return (to_c_x - from_c_x, to_c_y - from_c_y)

In [ ]:
cmap = plt.colormaps["nipy_spectral"]
figsize = (12, 12)
fontsize = 16
histo_res = 128
contour_res = 100
angle_elev = 30
angle_rot = -30
e_margin = 0.2
psd_margin = 0.2


x = [0, 1, 2, 3, 4]
y = [0, 1, 2, 3, 4]
_xmid = [xval + 0.5 for xval in x]
_ymid = [yval + 0.5 for yval in y]
x, y = np.meshgrid(x, y)
print(x)
print(y)
x, y = x.ravel(), y.ravel()
z = np.zeros_like(x)
dx = np.ones_like(x)
dy = np.ones_like(y)
dz = x + y

x = x + e_margin
y = y + psd_margin
dx = dx - e_margin
dy = dy - e_margin

xmid, ymid = np.meshgrid(_xmid, _ymid)
xmid, ymid = xmid.ravel(), ymid.ravel()
condition = xmid >= 2

mapped_colors = ["red" if meets_condition else "blue" for meets_condition in condition]

fig = plt.figure(figsize=figsize)
ax = plt.axes(projection='3d', computed_zorder=False)

ax.bar3d(x, y, z, dx, dy, dz, ec="black", lw=3, shade=False, color=mapped_colors)

text_path = make_text_path(["Test", "Text"], 1)
transform = mpl.transforms.Affine2D()
scaling_factor = (2, 1)
transform = transform.scale(*scaling_factor)
pos_from = get_bbox_center(text_path.get_extents(transform))
x_translate, y_translate = get_translation_to(pos_from, (7, 7))
transform = transform.translate(x_translate, y_translate)
x_center, y_center = get_bbox_center(text_path.get_extents(transform))
transform = transform.rotate_deg_around(x_center, y_center, 0)
text_path = transform.transform_path(text_path)
text_patch = convert_text_path_to_patch(text_path, 1)
patch_to_3d_plot_wall(text_patch, ax, "z")

ax.set_xlim(0, 10)
ax.set_ylim(0, 10)

# box = mpl.patches.Rectangle((1, 1), 1, 1, fc="#00000000")
# ax.add_patch(box)
# art3d.pathpatch_2d_to_3d(box, z=0, zdir="x")

# box = mpl.patches.FancyBboxPatch((2, 2), 1, 1, boxstyle="round", fc="#ffffff", zorder=0)
# ax.add_patch(box)
# art3d.pathpatch_2d_to_3d(box, z=0, zdir="x")

# textpath = mpl.text.TextPath((0, 0), "Test", size=0.5)
# trans = mpl.transforms.Affine2D().translate(2, 2)
# p1 = mpl.patches.PathPatch(trans.transform_path(textpath), zorder=1)
# ax.add_patch(p1)
# art3d.pathpatch_2d_to_3d(p1, z=0, zdir="x")

In [ ]:
# 3e 3D plot of neutron/gamma channels (AKA vaporwave island) (2025-04-24)
# cmap = plt.colormaps["nipy_spectral"]
cmap = plt.colormaps["viridis"]
figsize = (24, 24)
fontsize = 32
histo_res = 128
contour_res = 100
angle_elev = 30
angle_rot = -20
e_margin = 2
psd_margin = 0.002


for exp_name, data_dict in experiment_neutron_data.items():
    xe = data_dict[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = data_dict[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    dz = data_dict[ExperimentDataKey.PSD_HISTOGRAM]
    
    x, y = np.meshgrid(xe[:-1], ye[:-1])
    x, y = x.ravel(), y.ravel()
    z = np.full_like(x, 0)
    _dx = xe[1:] - xe[:-1]
    _dy = ye[1:] - ye[:-1]
    dx, dy = np.meshgrid(_dx, _dy)
    dz = dz.T
    dx, dy, dz = dx.ravel(), dy.ravel(), dz.ravel()

    x = x + e_margin
    dx = dx - e_margin
    y = y + psd_margin
    dy = dy - psd_margin
    
    height_mask = dz > 20
    x = x[height_mask]
    y = y[height_mask]
    z = z[height_mask]
    dx = dx[height_mask]
    dy = dy[height_mask]
    dz = dz[height_mask]
    
    fig = plt.figure(figsize=figsize)
    ax = plt.axes(projection='3d', computed_zorder=False)

    # min_dz = np.min(dz)
    min_dz = -2000
    max_dz = np.max(dz)
    norm = mpl.colors.Normalize(vmin=min_dz, vmax=max_dz)
    mapped_colors = [cmap(norm(dz_val)) for dz_val in dz]

    ax.view_init(angle_elev, angle_rot)
    ax.bar3d(x, y, z, dx, dy, dz,
             # color=color_alphas
             color=mapped_colors,
             shade=False,
             zsort="max",
             lw=0.2,
             ec="black"
            )

    bbox_params = {
        # "boxstyle": "round, pad=0.003, rounding_size=0.1",
        "fc": "white",
        "lw": 1,
        "fill": True,
        "alpha": 0.9
    }
    text_params = {
        "fontsize": fontsize-2,
        "bbox": bbox_params,
        "zorder": 5,
        "va": "center"
    }
    
    gamma_text = ["Gamma", "channel"]
    neutron_text = ["Neutron", "channel"]
    # scaling = (400, 0.025)
    scaling = (0.03, 280)
    gamma_position = (4400, 0.135)
    neutron_position = (4400, 0.35)
    # mutation_aspect = 3500 / 0.5
    # mutation_aspect = None
    linespacing = 1
    padding = 0.003
    rounding_size = 0.1
    font_properties = mpl.font_manager.FontProperties(weight="bold")

    for text_list, position in [
        (gamma_text, gamma_position),
        (neutron_text, neutron_position)
    ]:
        text_path = make_text_path(
            text_list,
            linespacing,
            # font_properties=font_properties
        )
        # bbox = make_bounding_box_for_text_path(
        #     text_path,
        #     padding,
        #     rounding_size,
        #     1,
        #     # mutation_aspect=mutation_aspect,
        #     **bbox_params
        # )
        transform = mpl.transforms.Affine2D()
        transform = transform.scale(*scaling)
        pos_from = get_bbox_center(text_path.get_extents(transform))
        x_translate, y_translate = get_translation_to(pos_from, position)
        transform = transform.translate(x_translate, y_translate)
        x_center, y_center = get_bbox_center(text_path.get_extents(transform))
        transform = transform.rotate_deg_around(x_center, y_center, 90)
        text_path = transform.transform_path(text_path)
        text_patch = convert_text_path_to_patch(text_path, 1)
        # patch_to_3d_plot_wall(bbox, ax, "z")
        patch_to_3d_plot_wall(text_patch, ax, "z")

    arrow_width = 0.04
    arrow_head_width = 0.07
    arrow_head_length = 400
    arrow_base_e = 3950
    arrow_len_psd = 0
    arrow_kwargs = {
        "width": arrow_width,
        "head_width": arrow_head_width,
        "head_length": arrow_head_length,
        "length_includes_head": True,
        "ec": "black",
        "fc": "none",
        "lw": 4
    }
    arrow_g = mpl.patches.FancyArrow(
        arrow_base_e, 0.135, -800, arrow_len_psd,
        **arrow_kwargs
    )
    patch_to_3d_plot_wall(arrow_g, ax, "z")
    arrow_n = mpl.patches.FancyArrow(
        arrow_base_e, 0.35, -2550, arrow_len_psd,
        **arrow_kwargs
    )
    patch_to_3d_plot_wall(arrow_n, ax, "z")

    # ax.set_title(f"{exp_name} PSD/Energy 3D Histogram", fontsize=fontsize+4)
    ax.set_xlabel("Energy (ADC channel x1000)", fontsize=fontsize)
    ax.set_ylabel("PSD", fontsize=fontsize)
    ax.set_zlabel("Counts (x1000)", fontsize=fontsize)
    ax.set_xlim(-100, 5000)
    ax.set_ylim(-0.01, 0.51)
    ax.set_zlim(0, 4000)
    ax.xaxis.set_major_formatter(lambda x, pos: f"{x / 1000:.1f}")
    ax.zaxis.set_major_formatter(lambda z, pos: f"{z / 1000:.1f}")
    ax.tick_params(labelsize=fontsize)
    ax.tick_params(pad=0)
    ax.tick_params(axis="x", pad=8)
    ax.tick_params(axis="y", pad=4)
    # ax.tick_params(axis="z", pad=-2)
    # for axis3d in [ax.xaxis, ax.yaxis, ax.zaxis]:
    #     axis3d.labelpad = 40
    ax.xaxis.labelpad = 48
    ax.yaxis.labelpad = 40
    ax.zaxis.labelpad = 44
    xaxis_ticklabels = ax.xaxis.get_ticklabels()
    for ticklabel in xaxis_ticklabels:
        ticklabel.set_ha("right")
        ticklabel.set_va("baseline")
    yaxis_ticklabels = ax.yaxis.get_ticklabels()
    for ticklabel in yaxis_ticklabels:
        ticklabel.set_ha("center")
        ticklabel.set_va("top")
    zaxis_ticklabels = ax.zaxis.get_ticklabels()
    for ticklabel in zaxis_ticklabels:
        ticklabel.set_ha("left")
        ticklabel.set_va("center_baseline")
    ax.xaxis.set_pane_color((1, 1, 1, 0))
    ax.yaxis.set_pane_color((1, 1, 1, 0))
    ax.zaxis.set_pane_color((1, 1, 1, 0))
    ax.set_box_aspect(None, zoom=0.85)

In [ ]:
a = np.arange(0, 200+1) * 2 - 1
v = np.arange(0, 200+20, 20)
idxs = np.searchsorted(a, v, side="right") - 1
print(a[idxs])

In [ ]:
testboolarray = np.full((10,), False)
idxs = [0, 2, 5, 7]
testboolarray[idxs] = True
print(testboolarray)

In [ ]:
# # 4a E slicing (2025-04-25)
# # cmap = plt.colormaps["nipy_spectral"]
# cmap = plt.colormaps["viridis"]
# figsize = (24, 24)
# fontsize = 32
# histo_res = 128
# contour_res = 100
# angle_elev = 30
# angle_rot = -20
# e_margin = 2
# psd_margin = 0.002


# for exp_name, data_dict in experiment_neutron_data.items():
#     xe = data_dict[ExperimentDataKey.HISTOGRAM_X_EDGES]
#     ye = data_dict[ExperimentDataKey.HISTOGRAM_Y_EDGES]
#     dz = data_dict[ExperimentDataKey.PSD_HISTOGRAM]
    
#     _dx = xe[1:] - xe[:-1]
#     _dy = ye[1:] - ye[:-1]

#     # TODO select ye values to get 1 E slice every x ADC (x=500?)
#     adc_step = 500
#     start_adc = 0
#     end_adc = 3000
#     selected_adcs = np.arange(start_adc, end_adc + adc_step, adc_step, dtype=float)
#     # for each value, find i s.t. ye[i] <= value and ye[i+1] > value
#     slice_indices = np.searchsorted(xe, selected_adcs, side="right") - 1
#     # use i values to make mask
#     mask = np.full_like(xe, False, dtype=bool)
#     mask[slice_indices] = True
#     # mask ye and use to make x/y meshgrid
#     xe = xe[mask]
#     _dx = _dx[mask[:-1]]
#     dz = dz.T
#     dz = dz[:, mask[:-1]]
    
#     x, y = np.meshgrid(xe, ye[:-1])
#     x, y = x.ravel(), y.ravel()
#     z = np.full_like(x, 0)
#     dx, dy = np.meshgrid(_dx, _dy)
#     print(dz.shape)
#     dx, dy, dz = dx.ravel(), dy.ravel(), dz.ravel()

#     x = x + e_margin
#     dx = dx - e_margin
#     y = y + psd_margin
#     dy = dy - psd_margin
    
#     height_mask = dz > 20
#     x = x[height_mask]
#     y = y[height_mask]
#     z = z[height_mask]
#     dx = dx[height_mask]
#     dy = dy[height_mask]
#     dz = dz[height_mask]
    
#     fig = plt.figure(figsize=figsize)
#     ax = plt.axes(projection='3d', computed_zorder=False)

#     # min_dz = np.min(dz)
#     min_dz = -2000
#     max_dz = np.max(dz)
#     norm = mpl.colors.Normalize(vmin=min_dz, vmax=max_dz)
#     mapped_colors = [cmap(norm(dz_val)) for dz_val in dz]

#     ax.view_init(angle_elev, angle_rot)
#     ax.bar3d(x, y, z, dx, dy, dz,
#              # color=color_alphas
#              color=mapped_colors,
#              shade=False,
#              zsort="max",
#              lw=0.2,
#              ec="black"
#             )

#     # bbox_params = {
#     #     # "boxstyle": "round, pad=0.003, rounding_size=0.1",
#     #     "fc": "white",
#     #     "lw": 1,
#     #     "fill": True,
#     #     "alpha": 0.9
#     # }
#     # text_params = {
#     #     "fontsize": fontsize-2,
#     #     "bbox": bbox_params,
#     #     "zorder": 5,
#     #     "va": "center"
#     # }
    
#     # gamma_text = ["Gamma", "channel"]
#     # neutron_text = ["Neutron", "channel"]
#     # # scaling = (400, 0.025)
#     # scaling = (0.03, 280)
#     # gamma_position = (4400, 0.135)
#     # neutron_position = (4400, 0.35)
#     # # mutation_aspect = 3500 / 0.5
#     # # mutation_aspect = None
#     # linespacing = 1
#     # padding = 0.003
#     # rounding_size = 0.1
#     # font_properties = mpl.font_manager.FontProperties(weight="bold")

#     # for text_list, position in [
#     #     (gamma_text, gamma_position),
#     #     (neutron_text, neutron_position)
#     # ]:
#     #     text_path = make_text_path(
#     #         text_list,
#     #         linespacing,
#     #         # font_properties=font_properties
#     #     )
#     #     # bbox = make_bounding_box_for_text_path(
#     #     #     text_path,
#     #     #     padding,
#     #     #     rounding_size,
#     #     #     1,
#     #     #     # mutation_aspect=mutation_aspect,
#     #     #     **bbox_params
#     #     # )
#     #     transform = mpl.transforms.Affine2D()
#     #     transform = transform.scale(*scaling)
#     #     pos_from = get_bbox_center(text_path.get_extents(transform))
#     #     x_translate, y_translate = get_translation_to(pos_from, position)
#     #     transform = transform.translate(x_translate, y_translate)
#     #     x_center, y_center = get_bbox_center(text_path.get_extents(transform))
#     #     transform = transform.rotate_deg_around(x_center, y_center, 90)
#     #     text_path = transform.transform_path(text_path)
#     #     text_patch = convert_text_path_to_patch(text_path, 1)
#     #     # patch_to_3d_plot_wall(bbox, ax, "z")
#     #     patch_to_3d_plot_wall(text_patch, ax, "z")

#     # arrow_width = 0.04
#     # arrow_head_width = 0.07
#     # arrow_head_length = 400
#     # arrow_base_e = 3950
#     # arrow_len_psd = 0
#     # arrow_kwargs = {
#     #     "width": arrow_width,
#     #     "head_width": arrow_head_width,
#     #     "head_length": arrow_head_length,
#     #     "length_includes_head": True,
#     #     "ec": "black",
#     #     "fc": "none",
#     #     "lw": 4
#     # }
#     # arrow_g = mpl.patches.FancyArrow(
#     #     arrow_base_e, 0.135, -800, arrow_len_psd,
#     #     **arrow_kwargs
#     # )
#     # patch_to_3d_plot_wall(arrow_g, ax, "z")
#     # arrow_n = mpl.patches.FancyArrow(
#     #     arrow_base_e, 0.35, -2550, arrow_len_psd,
#     #     **arrow_kwargs
#     # )
#     # patch_to_3d_plot_wall(arrow_n, ax, "z")

#     # # ax.set_title(f"{exp_name} PSD/Energy 3D Histogram", fontsize=fontsize+4)
#     # ax.set_xlabel("Energy (ADC channel x1000)", fontsize=fontsize)
#     # ax.set_ylabel("PSD", fontsize=fontsize)
#     # ax.set_zlabel("Counts (x1000)", fontsize=fontsize)
#     # ax.set_xlim(-100, 5000)
#     # ax.set_ylim(-0.01, 0.51)
#     # ax.set_zlim(0, 4000)
#     # ax.xaxis.set_major_formatter(lambda x, pos: f"{x / 1000:.1f}")
#     # ax.zaxis.set_major_formatter(lambda z, pos: f"{z / 1000:.1f}")
#     # ax.tick_params(labelsize=fontsize)
#     # ax.tick_params(pad=0)
#     # ax.tick_params(axis="x", pad=8)
#     # ax.tick_params(axis="y", pad=4)
#     # # ax.tick_params(axis="z", pad=-2)
#     # # for axis3d in [ax.xaxis, ax.yaxis, ax.zaxis]:
#     # #     axis3d.labelpad = 40
#     # ax.xaxis.labelpad = 48
#     # ax.yaxis.labelpad = 40
#     # ax.zaxis.labelpad = 44
#     # xaxis_ticklabels = ax.xaxis.get_ticklabels()
#     # for ticklabel in xaxis_ticklabels:
#     #     ticklabel.set_ha("right")
#     #     ticklabel.set_va("baseline")
#     # yaxis_ticklabels = ax.yaxis.get_ticklabels()
#     # for ticklabel in yaxis_ticklabels:
#     #     ticklabel.set_ha("center")
#     #     ticklabel.set_va("top")
#     # zaxis_ticklabels = ax.zaxis.get_ticklabels()
#     # for ticklabel in zaxis_ticklabels:
#     #     ticklabel.set_ha("left")
#     #     ticklabel.set_va("center_baseline")
#     # ax.xaxis.set_pane_color((1, 1, 1, 0))
#     # ax.yaxis.set_pane_color((1, 1, 1, 0))
#     # ax.zaxis.set_pane_color((1, 1, 1, 0))
#     # ax.set_box_aspect(None, zoom=0.85)

## Done!


In [ ]:
input("Processing done, hit Enter to finish")
helpers.stop()